# 330 — Anatomical parcellation classification (Yeo-7 & Yeo-17)

Decodes **which Yeo functional network** an electrode sits in, from its response profile
**concatenated across all three conditions** (`audio ⊕ picture ⊕ reading`). Run for each
feature variant × {Yeo-7, Yeo-17} × each classifier = 12 experiments.

The classes are imbalanced (some networks have far more electrodes than others), so
**balanced accuracy** and **macro-F1** are the headline metrics and `class_weight='balanced'`
is used throughout. Same nested-GroupKFold-by-patient protocol as 320 — so a network is only
"decodable" if it generalises across patients, not because a couple of patients dominate it.

See `390_results.ipynb` for the figures and how to read them.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_classify as C

# ---------------- config knobs ----------------
YEO_NETWORKS = (7, 17)
CLASSIFIERS  = ('logreg', 'rf')
OUTER_SPLITS = 5
INNER_SPLITS = 3
N_PERM       = 200
N_BOOT       = 1000
RANDOM_STATE = 42
print('yeo:', YEO_NETWORKS, '| classifiers:', CLASSIFIERS, '| n_perm:', N_PERM)


## Run all parcellation experiments
2 Yeo granularities × 3 variants × 2 classifiers = 12 runs, saved under
`outputs/classification/parcellation_yeo{7,17}/<variant>/<classifier>/runs/<id>/`.


In [ ]:
manifests = []
for n_net in YEO_NETWORKS:
    key = f'yeo{n_net}'
    for v in C.VARIANTS:
        X, y, groups, meta, cols = C.load_arrays('parcellation', key, v)
        for clf in CLASSIFIERS:
            m = C.run_experiment(f'parcellation_{key}', v, clf, X, y, groups, cols, meta,
                                 n_networks=n_net,
                                 outer_splits=OUTER_SPLITS, inner_splits=INNER_SPLITS,
                                 n_perm=N_PERM, n_boot=N_BOOT, random_state=RANDOM_STATE)
            manifests.append(m)
print('\ndone:', len(manifests), 'runs')


## Summary table


In [ ]:
df = C.list_runs()
df = df[df.task.str.startswith('parcellation')]
df[['task', 'variant', 'classifier', 'balanced_accuracy', 'chance_level',
    'macro_f1', 'permutation_p']].round(4)
